# 08 - Analisis SHAP (Explicabilidad)

**Objetivo**: Entender QUE hace que un memecoin sea una "joya".

SHAP (SHapley Additive exPlanations) nos dice:
- **Que features son mas importantes** para las predicciones
- **Como afecta cada feature** a la prediccion (positivo/negativo)
- **Por que un token especifico** fue clasificado como gem o failure

Visualizaciones:
1. Summary plot global (importancia de features)
2. Dependence plots (relacion feature -> prediccion)
3. Force plots individuales (explicar una prediccion)

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import joblib
import shap

from src.models.explainer import SHAPExplainer
import config

# Necesario para visualizaciones SHAP en notebooks
shap.initjs()
print("SHAP version:", shap.__version__)

## 1. Cargar modelo y datos

In [ ]:
# Cargar el mejor modelo (XGBoost)
model_path = config.MODELS_DIR / "xgboost.joblib"
if not model_path.exists():
    model_path = config.MODELS_DIR / "random_forest.joblib"

if model_path.exists():
    model = joblib.load(model_path)
    print(f"Modelo cargado: {model_path.name}")
else:
    print("No se encontraron modelos. Ejecuta notebook 07 primero.")
    model = None

In [ ]:
# Cargar features
try:
    features_df = pd.read_parquet(config.PROCESSED_DIR / "features.parquet")
    
    # Cargar nombres de features usados en entrenamiento
    feat_names_path = config.MODELS_DIR / "feature_names.json"
    if feat_names_path.exists():
        with open(feat_names_path) as f:
            feature_names = json.load(f)
        # Filtrar solo features usados en el modelo
        available = [f for f in feature_names if f in features_df.columns]
        X = features_df[available].copy()
    else:
        X = features_df.select_dtypes(include=[np.number]).copy()
    
    # Rellenar NaN con mediana
    X = X.fillna(X.median())
    
    print(f"Features cargados: {X.shape}")
except FileNotFoundError:
    print("No se encontro features.parquet. Ejecuta notebook 05.")
    X = pd.DataFrame()

## 2. Calcular SHAP values

In [ ]:
if model is not None and not X.empty:
    # Crear explainer
    explainer = SHAPExplainer(model=model, X_train=X)
    
    # Calcular SHAP values (puede tomar unos segundos)
    print("Calculando SHAP values...")
    shap_values = explainer.get_shap_values(X)
    print(f"SHAP values calculados: shape {shap_values.shape}")
else:
    print("Necesitas modelo y features")

## 3. Summary Plot Global

Muestra los features mas importantes ordenados por impacto.
- Cada punto es un token
- Posicion horizontal = impacto en prediccion
- Color = valor del feature (rojo=alto, azul=bajo)

In [ ]:
if 'explainer' in dir():
    explainer.plot_summary(X, max_display=15)
    plt.tight_layout()
    plt.show()

## 4. Top Features

In [ ]:
if 'explainer' in dir():
    top_features = explainer.get_top_features(X, n=15)
    print("Top 15 features por importancia SHAP:")
    display(top_features)

## 5. Dependence Plots

Muestra como cambia la prediccion cuando un feature cambia de valor.

In [ ]:
if 'explainer' in dir() and 'top_features' in dir():
    # Plot los top 4 features
    for feature_name in top_features['feature'].head(4):
        print(f"\n--- Dependence: {feature_name} ---")
        explainer.plot_dependence(X, feature=feature_name)
        plt.show()

## 6. Explicar Tokens Individuales

In [ ]:
if 'explainer' in dir() and not X.empty:
    # Explicar los primeros 3 tokens
    for i in range(min(3, len(X))):
        token_id = X.index[i] if X.index.name else f"Token {i}"
        print(f"\n{'='*50}")
        print(f"EXPLICACION: {token_id}")
        print(f"{'='*50}")
        
        explanation = explainer.explain_single_token(X, idx=i)
        
        if explanation:
            print(f"\nPrediccion: {explanation.get('prediction', 'N/A')}")
            
            print("\nFactores que EMPUJAN hacia gem:")
            for feat, val in explanation.get('top_positive', []):
                print(f"  + {feat}: {val:.4f}")
            
            print("\nFactores que EMPUJAN hacia failure:")
            for feat, val in explanation.get('top_negative', []):
                print(f"  - {feat}: {val:.4f}")

## 7. Guardar SHAP values

In [ ]:
if 'shap_values' in dir():
    # Guardar SHAP values como parquet
    shap_df = pd.DataFrame(shap_values, columns=X.columns, index=X.index)
    shap_df.to_parquet(config.MODELS_DIR / "shap_values.parquet")
    print(f"SHAP values guardados: {config.MODELS_DIR / 'shap_values.parquet'}")
    
    # Guardar top features como JSON
    if 'top_features' in dir():
        top_feat_dict = top_features.to_dict('records')
        with open(config.MODELS_DIR / "top_features.json", 'w') as f:
            json.dump(top_feat_dict, f, indent=2)
    
    print("\n✅ Analisis SHAP completado!")
    print("Ahora puedes ver todo en el dashboard: streamlit run dashboard/app.py")

## Resumen: Que hace que un memecoin sea una joya?

Basado en el analisis SHAP, los factores mas importantes son:

1. (Completar despues de ejecutar el analisis)
2. 
3. 

### Interpretacion:
- (Completar con hallazgos especificos)